### Buidling Chatbot with multiple tools using Langgraph

AIM: Creata a chatbot with tool capabilities from  wikipedia search and some functions

In [ ]:
from langchain_community.tools import WikipediaQueryRun

from langchain_community.utilities import WikipediaAPIWrapper

In [ ]:
api_wrapper_arxiv = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_char_limit=500
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper_arxiv)

print(wiki_tool.invoke("Attention is all you need"))

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
# TAVILY Search Tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search_tool = TavilySearchResults()

print(tavily_search_tool.invoke("Provide me the recent SpaceX news"))

In [ ]:
### Combine all the tools
tools =[wiki_tool,tavily_search_tool]

In [ ]:
# Intialize chat model

from langchain_groq import ChatGroq

llm_groq = ChatGroq(model="qwen/qwen3-32b")

llm_with_tools = llm_groq.bind_tools(tools)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from pprint import pprint

llm_with_tools.invoke([HumanMessage(content="who own SpaceX")])

In [ ]:
# State Schema

from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
class State(TypedDict):
    messages:Annotated[list[AnyMessage], add_messages]

In [ ]:
# Node definition

def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

In [ ]:
### Entire Chatbot with LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

# Create nodes
builder.add_node("llm_node", tool_calling_llm)
builder.add_node("tools",ToolNode(tools)) 

# Add edges
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", tools_condition)
builder.add_edge("tools", END)

graph_builder = builder.compile()

# View the graph
display(Image(graph_builder.get_graph().draw_mermaid_png()))




In [ ]:

messages=graph_builder.invoke({"messages":HumanMessage(content="what is the current weather in Dallas?")}) # LLM decides to call the Tavily tool 

for m in messages['messages']:
    m.pretty_print()

  


In [ ]:
messages=graph_builder.invoke({"messages":HumanMessage(content="what is Attention Is all you need?")})  # LLM decides to call the wikipedia tool 

for m in messages['messages']:
    m.pretty_print()

  